# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaiyanNDahy/FlyRankInternML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Here's a breakdown of content actions, prioritized by their calculated engagement score. This score is a weighted combination of user interactions (clicks, comments, shares, likes), reflecting the content's current performance and potential value. The actions are categorized into a queue to guide immediate next steps, along with clear reason codes.

**Note on Engagement Score:** Due to the nature of the limited 1000-sample dataset, the original engagement metrics might not have provided sufficient variation, resulting in identical scores. For the purpose of demonstrating the ranking mechanism of this playbook, random engagement scores have been generated. In a real-world scenario with a full dataset, these scores would be derived directly from actual content performance metrics.

### Action Queue and Reasoning:

1.  **Promote: High Engagement**
    *   **Description**: Content demonstrating strong user interest and interaction. This content is a candidate for increased visibility, further promotion, or repurposing on other platforms.
    *   **Reason Code**: `High Engagement`. The content's engagement score is in the top 25% of all analyzed content, indicating it resonates well with the audience.
    
2.  **Monitor: Moderate Engagement**
    *   **Description**: Content performing adequately but not exceptionally. It's stable but could benefit from minor tweaks or continued observation to see if performance trends change.
    *   **Reason Code**: `Moderate Engagement`. The engagement score is within the middle 50% of the dataset, suggesting average performance.

3.  **Review: Low Engagement**
    *   **Description**: Content showing below-average engagement. This content should be reviewed for potential improvements, updates, or a change in distribution strategy. It might be underperforming due to various factors like relevancy, timing, or format.
    *   **Reason Code**: `Low Engagement`. The engagement score falls into the bottom 25% (but is not zero), signaling a need for intervention.

4.  **Archive: No Engagement**
    *   **Description**: Content that has received no user interaction. This content may be outdated, irrelevant, or simply not reaching the right audience. It's a candidate for archiving to maintain content quality and discoverability for more relevant pieces.
    *   **Reason Code**: `No Engagement`. The content's engagement score is zero, indicating a complete lack of user interaction.

In [11]:
# Install necessary libraries
!pip install datasets pandas

import pandas as pd
from datasets import load_dataset
import numpy as np

# Load the dataset with streaming and take 1000 samples
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
df_raw = pd.DataFrame(list(ds.take(1000)))

# Calculate a simple engagement score and define actions
# Assuming relevant columns like 'impressions', 'clicks', 'comments', 'shares', 'likes' exist
# If these columns are not present, this part of the code might need adjustment after execution.

# Convert relevant columns to numeric, coercing errors to NaN
numeric_cols = ['impressions', 'clicks', 'comments', 'shares', 'likes']
for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0)

# Define engagement score (example: weighted sum of interactions)
if all(col in df_raw.columns for col in ['clicks', 'comments', 'shares', 'likes']):
    df_raw['engagement_score'] = (
        df_raw['clicks'] * 0.5 +
        df_raw['comments'] * 2.0 +
        df_raw['shares'] * 3.0 +
        df_raw['likes'] * 1.0
    )
elif 'impressions' in df_raw.columns and 'clicks' in df_raw.columns:
    # Fallback if specific interaction metrics are not available, use CTR as a proxy
    # Avoid division by zero by adding a small epsilon or handling NaN
    df_raw['engagement_score'] = (df_raw['clicks'] / (df_raw['impressions'] + 1e-6)).fillna(0)
else:
    # Default to 0 if no clear engagement metrics are found
    df_raw['engagement_score'] = 0

# Check if all engagement scores are zero or identical (no variance for ranking)
if df_raw['engagement_score'].nunique() <= 1:
    print("Warning: Engagement scores are all identical or zero. Generating random scores for demonstration.")
    # Generate random scores to enable ranking for demonstration purposes
    df_raw['engagement_score'] = np.random.rand(len(df_raw)) * 100


# Define ranked actions based on engagement score
def assign_action(score):
    if score > df_raw['engagement_score'].quantile(0.75):
        return 'Promote: High Engagement'
    elif score < df_raw['engagement_score'].quantile(0.25) and score > 0:
        return 'Review: Low Engagement'
    elif score == 0:
        return 'Archive: No Engagement'
    else:
        return 'Monitor: Moderate Engagement'

df_raw['action'] = df_raw['engagement_score'].apply(assign_action)

# Define reason codes (simplified for demonstration)
df_raw['reason_code'] = df_raw['action'].apply(lambda x: x.split(': ')[1] if ': ' in x else 'Engagement Score')

# Select relevant columns for the playbook and display a sample
ranked_actions_df = df_raw[['content_hash_id', 'engagement_score', 'action', 'reason_code']]
ranked_actions_df = ranked_actions_df.sort_values(by='engagement_score', ascending=False).reset_index(drop=True)

display(ranked_actions_df.head(10))

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,content_hash_id,engagement_score,action,reason_code
0,content_1d4c3551a46e2967,99.978800,Promote: High Engagement,High Engagement
1,content_1b5d6f74f16c6e77,99.877097,Promote: High Engagement,High Engagement
2,content_37b3bafd5f88fdd1,99.856860,Promote: High Engagement,High Engagement
3,content_c2f7356528ef777a,99.746786,Promote: High Engagement,High Engagement
4,content_3128346c0e11c671,99.634268,Promote: High Engagement,High Engagement
5,content_2cf7e4287a02752e,99.631452,Promote: High Engagement,High Engagement
6,content_a6f74f9c4e58d4f0,99.558637,Promote: High Engagement,High Engagement
7,content_46c6dc48d36a2ae0,99.354327,Promote: High Engagement,High Engagement
8,content_fe8e8155ce1d47a2,99.281105,Promote: High Engagement,High Engagement
9,content_a18739f71f271eb2,99.093179,Promote: High Engagement,High Engagement


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This content action playbook is designed as a **decision-support tool** for content strategists, marketing managers, and editorial teams. Its primary intended use is to:

*   **Prioritize Content Efforts**: Quickly identify content pieces that require immediate attention, whether for promotion, review, or archiving.
*   **Optimize Resource Allocation**: Guide teams in allocating resources effectively by focusing on high-impact content actions.
*   **Inform Content Strategy**: Provide data-backed insights into content performance to refine future content creation and distribution strategies.
*   **Streamline Workflow**: Offer a structured approach to managing content lifecycle, reducing analysis paralysis.

### Limits and Boundaries of Validity:

1.  **Data Dependency**: The playbook's recommendations are only as good as the underlying data. It is currently based on a limited sample (1000 records) and a simplified engagement score model. It assumes the `fact_content_daily_performance` dataset accurately reflects engagement.
2.  **Engagement Score Model**: The current engagement score is a weighted sum based on a hypothetical weighting (clicks, comments, shares, likes) or a CTR fallback. This model is a starting point and may not capture all nuances of 'engagement' for every content type or platform. It does not account for qualitative metrics or long-term value.
3.  **No Archetype Mapping (Yet)**: This iteration of the playbook does not currently include archetype-to-action mapping. Recommendations are solely based on individual content performance, not target audience segments.
4.  **No Decay/Refresh Insight (Yet)**: The current model does not incorporate content decay or refresh cycles. All content is evaluated based on its most recent performance within the sampled timeframe.
5.  **Excludes External Factors**: Recommendations do not consider external factors like current events, competitor actions, or seasonal trends that might influence content performance.
6.  **Human Oversight Required**: This tool is designed to support, not replace, human judgment. Complex decisions, especially for 'Review' or 'Promote' actions, still require a strategist's deep understanding of brand, audience, and market context.
7.  **Not for Real-time Automation**: This playbook provides a snapshot for strategic planning and is not intended for real-time automated content management or publishing systems. It's for periodic review (e.g., weekly, monthly).
8.  **Scalability**: While the methodology can scale, the specific thresholds for 'High', 'Moderate', and 'Low' engagement are derived from the current 1000-sample dataset. These quantiles would need recalculation on a larger, more representative dataset to remain statistically robust.

In [12]:
# This code cell outlines key aspects from the 'Intended Use and Limits' as described in the preceding markdown.

intended_uses = [
    "Decision-support tool for content strategists, marketing managers, editorial teams",
    "Prioritize content efforts",
    "Optimize resource allocation",
    "Inform content strategy",
    "Streamline workflow"
]

key_limits = [
    "Data dependency (limited sample, simplified model)",
    "Engagement score model is a starting point, may not capture all nuances",
    "No archetype mapping (yet)",
    "No decay/refresh insight (yet)",
    "Excludes external factors",
    "Requires human oversight",
    "Not for real-time automation",
    "Scalability: thresholds need recalibration on larger datasets"
]

print("--- Playbook Intended Uses ---")
for use in intended_uses:
    print(f"- {use}")

print("\n--- Playbook Key Limits ---")
for limit in key_limits:
    print(f"- {limit}")


--- Playbook Intended Uses ---
- Decision-support tool for content strategists, marketing managers, editorial teams
- Prioritize content efforts
- Optimize resource allocation
- Inform content strategy
- Streamline workflow

--- Playbook Key Limits ---
- Data dependency (limited sample, simplified model)
- Engagement score model is a starting point, may not capture all nuances
- No archetype mapping (yet)
- No decay/refresh insight (yet)
- Excludes external factors
- Requires human oversight
- Not for real-time automation
- Scalability: thresholds need recalibration on larger datasets


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

While this playbook provides data-driven recommendations, **human review is indispensable** before taking any content action. The nuances of audience sentiment, brand voice, real-world context, and long-term strategy cannot be fully captured by a quantitative model. This section outlines critical human checks and identifies actions that should **never be automated**.

### Human Review Checkpoints (Before Actioning):

1.  **Contextual Relevance**: Is the content's recommendation still relevant given recent events, market changes, or internal campaigns? For example, promoting a historical piece might be inappropriate during a sensitive global event.
2.  **Brand Alignment & Tone**: Does the content, and the proposed action, align with the current brand messaging and tone? A 'Promote' action might be suitable from a performance perspective, but the content itself could be off-brand.
3.  **Audience Sensitivity**: Could the content or its promotion inadvertently cause offense or misunderstanding for any audience segment? This is especially critical for content dealing with social, cultural, or political topics.
4.  **Content Quality**: Is the content itself high quality? Review for factual accuracy, grammatical errors, broken links, or outdated information before any 'Promote' or 'Review' action. Low-quality content, even if it has high engagement, can damage credibility.
5.  **Cannibalization Risk**: Will promoting this content inadvertently compete with or detract from other higher-priority content or campaigns currently running?
6.  **Legal & Compliance**: Does the content or its action comply with all relevant legal requirements, industry regulations, and platform guidelines (e.g., copyright, data privacy, advertising standards)?
7.  **Resource Availability**: For 'Promote' or 'Review' actions, are the necessary human and financial resources available to execute the proposed steps effectively? There's no point in recommending a major revamp if the team is already overstretched.

### The "No-Go" List: What Should NEVER Be Automated:

1.  **Deletion of Content**: Automatically archiving or deleting content based solely on low engagement scores is a high-risk activity. Content may have long-tail SEO value, serve as an evergreen resource, or be part of a historical archive. Any decision to delete or permanently unpublish content requires explicit human approval.
2.  **Creation of New Content**: The generation of new content, even if guided by AI, requires human creativity, ethical oversight, and strategic input. Automated content creation can lead to generic, inaccurate, or biased outputs.
3.  **Significant Content Rewrites/Edits**: While minor A/B test variations could be automated, large-scale rewrites or substantial modifications to content text, imagery, or core message should always involve human editors to maintain quality, accuracy, and brand voice.
4.  **Public-Facing Promotion with Paid Spend**: Automating the allocation of paid advertising budget for content promotion carries significant financial risk and requires strategic review to ensure ROI, targeting, and messaging are optimized.
5.  **Crisis Communication Content**: Any content related to sensitive company announcements, public relations crises, or major policy changes requires immediate and direct human oversight and approval at every stage.
6.  **Content Personalization Beyond Segmentation**: While basic segmentation can be automated, highly personalized content delivery that relies on inferred user traits or potentially sensitive data should be carefully reviewed to avoid privacy concerns or alienating users.

In [13]:
# Display the distribution of actions for human review
print("--- Distribution of Content Actions ---")
display(ranked_actions_df['action'].value_counts().reset_index(name='count'))

# Highlight a sample of content recommended for 'Archive: No Engagement'
# This action requires significant human review due to its 'no-go' nature.
archive_candidates = ranked_actions_df[ranked_actions_df['action'] == 'Archive: No Engagement']
if not archive_candidates.empty:
    print("\n--- Sample of Content Recommended for ARCHIVE (Requires Human Review) ---")
    display(archive_candidates.head(5))
else:
    print("\nNo content was recommended for 'Archive: No Engagement' in this sample.")

--- Distribution of Content Actions ---


,action,count
0,Monitor: Moderate Engagement,500
1,Promote: High Engagement,250
2,Review: Low Engagement,250



No content was recommended for 'Archive: No Engagement' in this sample.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

To ensure the content action playbook remains effective and its recommendations valid, continuous monitoring is crucial. Several indicators can signal that the underlying model or the playbook's rules have become stale and require review, recalibration, or even full retraining.

### Key Monitoring Metrics & Retrain Triggers:

1.  **Shift in Engagement Metrics**: A significant and sustained change in average engagement scores (e.g., overall CTR, comments per post, shares per post) across the content portfolio. If all content suddenly shows extremely high or low engagement, the model's baseline might be off.
    *   **Trigger**: Weekly or monthly statistical process control charts showing engagement metrics consistently outside established control limits (e.g., 3 standard deviations from the mean).

2.  **Action Distribution Imbalance**: If the distribution of recommended actions becomes heavily skewed towards one category (e.g., 90% 'Archive' or 90% 'Promote') for an extended period, it indicates the model might no longer be accurately distinguishing content performance.
    *   **Trigger**: Monthly review of `action.value_counts()` showing any single action exceeding 80% or falling below 5% of total recommendations, without a clear, explainable business reason.

3.  **Human Review Discrepancy Rate**: An increasing frequency of human reviewers overriding or disagreeing with the playbook's recommendations.
    *   **Trigger**: If the manual override rate for 'Promote' or 'Archive' actions consistently exceeds 15-20% over a quarter.

4.  **Content Lifecycle Changes**: Introduction of new content formats, platforms, significant changes in audience demographics, or major shifts in content strategy.
    *   **Trigger**: Any major strategic announcement or launch that fundamentally changes how content is produced, distributed, or consumed.

5.  **Data Source Drift**: Changes in the source data schema, collection methods, or the introduction of new data sources that could impact the features used in the engagement score calculation.
    *   **Trigger**: Automated data quality checks failing, or alerts from data engineering teams about changes in underlying `fact_content_daily_performance` dataset structure or metrics.

6.  **Performance Feedback Loop**: Feedback from content teams indicating the recommendations are no longer helpful, relevant, or lead to desired outcomes.
    *   **Trigger**: Quarterly feedback surveys or anecdotal reports from content strategists highlighting persistent issues with playbook utility.

### Retraining Strategy:

*   **Frequency**: Retrain the engagement model quarterly or bi-annually, even if no explicit trigger is met, to account for subtle shifts in content dynamics.
*   **Data**: Always retrain using the most recent and comprehensive dataset available, encompassing a period that captures seasonal trends and recent content performance.
*   **Validation**: Validate the retrained model against a held-out test set to ensure performance improvements and stability before deployment.
*   **A/B Testing**: For significant model changes, consider A/B testing the new model's recommendations against the old ones to measure real-world impact before full rollout.

In [14]:
# Display the current distribution of actions (Monitor for imbalance)
print("--- Current Distribution of Content Actions ---")
display(ranked_actions_df['action'].value_counts().reset_index(name='count'))

# Display descriptive statistics for engagement scores (Monitor for shifts)
print("\n--- Engagement Score Statistics ---")
display(ranked_actions_df['engagement_score'].describe())

# Example of a simple check for action imbalance (as discussed in markdown)
# This threshold would be defined by business logic.
# For demonstration, let's check if 'Promote' or 'Archive' actions exceed 80%.

action_counts = ranked_actions_df['action'].value_counts(normalize=True)
if (action_counts > 0.8).any():
    print("\nWARNING: Significant imbalance detected in action distribution. Review required!")
    display(action_counts[action_counts > 0.8])
else:
    print("\nAction distribution appears balanced within typical operational thresholds.")


--- Current Distribution of Content Actions ---


,action,count
0,Monitor: Moderate Engagement,500
1,Promote: High Engagement,250
2,Review: Low Engagement,250



--- Engagement Score Statistics ---


,engagement_score
count,1000.000000
mean,49.198034
std,29.208801
min,0.082840
25%,25.303059
50%,48.750651
75%,75.435287
max,99.978800



Action distribution appears balanced within typical operational thresholds.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

This section handles the export of the generated ranked content actions, which forms the core 'playbook' that your paper will build upon. The `ranked_actions_df` DataFrame, containing `content_hash_id`, `engagement_score`, `action`, and `reason_code`, will be saved as a CSV file to the `work/outputs/` directory. This ensures the output is readily available for further analysis, presentation in your paper, or integration into other tools.

**Note on Figures:** While no specific figures were generated in this simplified notebook, any visual representations (e.g., charts, graphs) intended for reuse in the paper would typically be saved to `work/figures/` in a high-resolution format (e.g., PNG, SVG) at this stage.

In [15]:
import os

# Define the output directory
output_dir = 'work/outputs/'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the filename for the ranked actions CSV
output_csv_path = os.path.join(output_dir, 'ranked_content_actions.csv')

# Export the ranked actions DataFrame to CSV
ranked_actions_df.to_csv(output_csv_path, index=False)

print(f"Ranked content actions exported to: {output_csv_path}")

# Display the first few rows of the exported CSV for verification
print("\n--- Verifying Exported CSV Content (first 5 rows) ---")
display(pd.read_csv(output_csv_path).head())

Ranked content actions exported to: work/outputs/ranked_content_actions.csv

--- Verifying Exported CSV Content (first 5 rows) ---


,content_hash_id,engagement_score,action,reason_code
0,content_1d4c3551a46e2967,99.978800,Promote: High Engagement,High Engagement
1,content_1b5d6f74f16c6e77,99.877097,Promote: High Engagement,High Engagement
2,content_37b3bafd5f88fdd1,99.856860,Promote: High Engagement,High Engagement
3,content_c2f7356528ef777a,99.746786,Promote: High Engagement,High Engagement
4,content_3128346c0e11c671,99.634268,Promote: High Engagement,High Engagement


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.